# 02 — Feature Engineering

**Goal:** Construct behavioral and structural features from raw customer attributes. Split into two pipeline branches:
- `df_logreg` — OHE, no multicollinear features (for linear baseline)
- `df_catboost` — raw string categoricals, full feature set (for production model)

> **Architecture note:** In the original production pipeline, features were velocity-based time-series ratios computed from 11 monthly ClickHouse snapshots (V1, V3, Acceleration). Here we reconstruct the same behavioral signal logic using `tenure`, `MonthlyCharges`, and service subscription patterns as cross-sectional proxies.

In [ ]:
import numpy as np
import pandas as pd
import json
import os

pd.set_option('display.max_columns', None)

DATA_INTERIM  = '../data/interim/df_base_clean.parquet'
OUT_LOGREG    = '../data/interim/df_logreg.parquet'
OUT_CATBOOST  = '../data/interim/df_catboost.parquet'
CONTRACT_PATH = '../notebooks/feature_lists.json'

df = pd.read_parquet(DATA_INTERIM)
print(f'Loaded: {df.shape}')
df.head(2)

## Block A — Service Intensity Ratios

Proxy for the **loyalty / engagement indices** from the production pipeline.
High service adoption → higher switching cost → lower churn probability.

In [ ]:
# Count of value-added services subscribed
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']

for col in service_cols:
    df[col + '_flag'] = (df[col] == 'Yes').astype(int)

flag_cols = [c + '_flag' for c in service_cols]
df['Num_Services']       = df[flag_cols].sum(axis=1)
df['Streaming_Ratio']    = (df['StreamingTV_flag'] + df['StreamingMovies_flag']) / (df['Num_Services'] + 1e-6)
df['Protection_Ratio']   = (df['OnlineSecurity_flag'] + df['DeviceProtection_flag'] + df['TechSupport_flag']) / (df['Num_Services'] + 1e-6)
df['Has_Any_Service']    = (df['Num_Services'] > 0).astype(int)
df['Is_FullBundle']      = (df['Num_Services'] == 6).astype(int)

print('Service features created.')
print(df[['Num_Services', 'Streaming_Ratio', 'Protection_Ratio']].describe().round(3))

## Block B — Tenure & Loyalty Signals

Proxy for **lifetime stage** and **switching cost**. Short tenure = high churn risk window.

In [ ]:
# Lifecycle stage flags (mirroring Is_Newbie / lifetime thresholds from production)
df['Is_NewCustomer']    = (df['tenure'] <= 6).astype(int)    # < 6 months: highest risk window
df['Is_Established']    = (df['tenure'].between(7, 24)).astype(int)
df['Is_Loyal']          = (df['tenure'] > 24).astype(int)

# Tenure buckets (useful for CatBoost as ordinal groupings)
df['Tenure_Band'] = pd.cut(
    df['tenure'],
    bins=[0, 6, 12, 24, 48, 72],
    labels=['0-6m', '6-12m', '12-24m', '24-48m', '48-72m']
).astype(str)

# Contract risk encoding (Month-to-month = highest churn risk)
contract_risk = {'Month-to-month': 2, 'One year': 1, 'Two year': 0}
df['Contract_Risk'] = df['Contract'].map(contract_risk)

print('Tenure & loyalty features created.')

## Block C — Charge Dynamics (Bill Shock Proxies)

In the production pipeline, **Bill Shock** was detected via `REVENUE_TOTAL` velocity:
if monthly charges jumped relative to 3-month average, churn risk spiked.
Here we derive equivalent signals from static charge fields.

In [ ]:
# Implied average monthly spend over lifetime
df['Avg_Lifetime_Charge'] = df['TotalCharges'] / (df['tenure'] + 1e-6)

# Bill shock index: current monthly vs lifetime average
# > 1.0 means they're paying more now than historical average → churn signal
df['Bill_Shock_Index'] = (df['MonthlyCharges'] / (df['Avg_Lifetime_Charge'] + 1e-6)).clip(0, 5)

# Charge per service — high cost relative to services subscribed
df['Charge_Per_Service'] = df['MonthlyCharges'] / (df['Num_Services'] + 1)

# High spender flag (above 75th percentile)
charge_p75 = df['MonthlyCharges'].quantile(0.75)
df['Is_HighSpender'] = (df['MonthlyCharges'] > charge_p75).astype(int)

print(f'Bill shock index — mean: {df["Bill_Shock_Index"].mean():.3f}, max: {df["Bill_Shock_Index"].max():.3f}')
print(f'Charge per service — mean: {df["Charge_Per_Service"].mean():.2f}')

## Block D — Payment & Digital Risk Flags

In [ ]:
# Electronic check is the highest-churn payment method (from EDA)
df['Is_ElectronicCheck'] = (df['PaymentMethod'] == 'Electronic check').astype(int)

# Paperless billing combined with electronic check = maximum digital friction / churn risk
df['Digital_Risk_Combo'] = df['Is_ElectronicCheck'] * df['PaperlessBilling']

# No phone service flag
df['No_PhoneService'] = (df['MultipleLines'] == 'No phone service').astype(int)

# Fiber optic flag (highest churn internet type from EDA)
df['Is_FiberOptic'] = (df['InternetService'] == 'Fiber optic').astype(int)

print('Payment & digital risk flags created.')
print(df[['Is_ElectronicCheck', 'Digital_Risk_Combo', 'Is_FiberOptic']].mean().round(3))

## Pipeline Split — Two Branches

Following the same dual-branch architecture as the production pipeline:
- **LogReg branch**: OHE for categoricals, drop multicollinear features, drop ratio-heavy features that cause regularization issues
- **CatBoost branch**: raw string categoricals (CatBoost handles them natively), full feature set

In [ ]:
service_id_cols  = ['customerID']
target_col       = ['Churn']
cat_cols         = ['InternetService', 'Contract', 'PaymentMethod', 'MultipleLines', 'Tenure_Band']

# Drop original service Yes/No text columns (replaced by _flag versions)
original_service_text = service_cols  # ['OnlineSecurity', 'OnlineBackup', ...]
df = df.drop(columns=original_service_text)

# --- BRANCH 1: LOGISTIC REGRESSION ---
# Drop multicollinear features and ratio-heavy features
# that cause instability in linear models
logreg_drop = [
    'TotalCharges',          # = MonthlyCharges * tenure (perfect multicollinearity)
    'Avg_Lifetime_Charge',   # derived from TotalCharges
    'Bill_Shock_Index',      # non-linear ratio, poor in linear models
    'Is_ElectronicCheck',    # captured by OHE of PaymentMethod
    'Tenure_Band',           # redundant with Is_NewCustomer / Is_Loyal flags
    'Contract_Risk',         # redundant with OHE of Contract
]

df_logreg = df.drop(columns=logreg_drop + service_id_cols, errors='ignore')
df_logreg = pd.get_dummies(
    df_logreg,
    columns=[c for c in cat_cols if c in df_logreg.columns and c != 'Tenure_Band'],
    drop_first=True  # avoid dummy variable trap
)

print(f'LogReg branch shape: {df_logreg.shape}')
print(f'Churn column present: {"Churn" in df_logreg.columns}')

In [ ]:
# --- BRANCH 2: CATBOOST ---
# Keep raw string categoricals — CatBoost handles them natively (no OHE needed)
# Keep full feature set including ratios and interaction terms
catboost_drop = service_id_cols  # only drop ID column

df_catboost = df.drop(columns=catboost_drop, errors='ignore')

# Ensure categoricals are string type (CatBoost requirement)
for col in cat_cols:
    if col in df_catboost.columns:
        df_catboost[col] = df_catboost[col].astype(str)

print(f'CatBoost branch shape: {df_catboost.shape}')
print(f'Churn column present: {"Churn" in df_catboost.columns}')

## Feature Contract (JSON)

Locking feature lists at engineering time prevents train/inference skew.
The same JSON is loaded by notebooks 03 and 05.

In [ ]:
feature_contract = {
    'X_logreg_cols':   [c for c in df_logreg.columns  if c != 'Churn'],
    'X_catboost_cols': [c for c in df_catboost.columns if c != 'Churn'],
    'cat_features':    [c for c in cat_cols if c in df_catboost.columns]
}

with open(CONTRACT_PATH, 'w') as f:
    json.dump(feature_contract, f, indent=4)

print(f'Feature contract saved: {CONTRACT_PATH}')
print(f'LogReg features : {len(feature_contract["X_logreg_cols"])}')
print(f'CatBoost features: {len(feature_contract["X_catboost_cols"])}')
print(f'Categorical cols : {feature_contract["cat_features"]}')

In [ ]:
df_logreg.to_parquet(OUT_LOGREG, index=False)
df_catboost.to_parquet(OUT_CATBOOST, index=False)
print(f'Saved df_logreg   → {OUT_LOGREG}')
print(f'Saved df_catboost → {OUT_CATBOOST}')
print('\nFinished 02_feature_engineering.ipynb')